In [ ]:
!pip install -q seqeval
!pip install -q pymupdf
!pip install -q transformers datasets evaluate jiwer Pillow torch torchvision
!pip install -q detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu118/torch2.0/index.html 2>/dev/null || echo "detectron2 optional, skipping"

print("All dependencies installed.")

In [ ]:
import os
import json
import fitz          # PyMuPDF
import torch
import evaluate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from PIL import Image, ImageDraw, ImageFont
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR
from tqdm import tqdm
from datasets import load_dataset
from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification,
    LayoutLMv3Config
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
print("Loading FUNSD dataset...")
funsd = load_dataset("nielsr/funsd-layoutlmv3")

print(f"Dataset: {funsd}")
print(f"\nTrain samples: {len(funsd['train'])}")
print(f"Test samples:  {len(funsd['test'])}")

# Inspect one sample
sample = funsd['train'][0]
print(f"\nSample keys: {list(sample.keys())}")
print(f"Image type:  {type(sample['image'])}")
print(f"Image size:  {sample['image'].size}")
print(f"Num words:   {len(sample['tokens'])}")
print(f"First 5 words:  {sample['tokens'][:5]}")
print(f"First 5 boxes:  {sample['bboxes'][:5]}")
print(f"First 5 labels: {sample['ner_tags'][:5]}")

In [ ]:
LABEL_LIST = ['O', 'B-QUESTION', 'I-QUESTION', 'B-ANSWER', 'I-ANSWER', 'B-HEADER', 'I-HEADER']
LABEL2ID   = {label: idx for idx, label in enumerate(LABEL_LIST)}
ID2LABEL   = {idx: label for label, idx in LABEL2ID.items()}
NUM_LABELS = len(LABEL_LIST)

# Color mapping for visualization
LABEL_COLORS = {
    'O':          '#CCCCCC',   # gray
    'B-QUESTION': '#FF6B6B',   # red
    'I-QUESTION': '#FFB3B3',   # light red
    'B-ANSWER':   '#4ECDC4',   # teal
    'I-ANSWER':   '#A8E6CF',   # light teal
    'B-HEADER':   '#FFE66D',   # yellow
    'I-HEADER':   '#FFF3B0',   # light yellow
}

print("Label space:")
for label, idx in LABEL2ID.items():
    print(f"  {idx}: {label}")

print(f"\nTotal labels: {NUM_LABELS}")

In [ ]:
def visualize_sample(sample, id2label, label_colors, title="FUNSD Sample"):
    """
    Draw bounding boxes on document image, colored by label.
    """
    image = sample['image'].convert('RGB')
    draw  = ImageDraw.Draw(image, 'RGBA')
    
    words  = sample['tokens']
    boxes  = sample['bboxes']
    labels = sample['ner_tags']
    
    for word, box, label_id in zip(words, boxes, labels):
        label = id2label[label_id]
        color = label_colors.get(label, '#CCCCCC')
        
        # Convert hex color to RGBA with transparency
        r = int(color[1:3], 16)
        g = int(color[3:5], 16)
        b = int(color[5:7], 16)
        
        x0, y0, x1, y1 = box
        draw.rectangle([x0, y0, x1, y1], fill=(r, g, b, 100), outline=(r, g, b, 220), width=2)
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 12))
    ax.imshow(image)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')
    
    # Legend
    legend_items = [
        patches.Patch(color='#FF6B6B', label='QUESTION'),
        patches.Patch(color='#4ECDC4', label='ANSWER'),
        patches.Patch(color='#FFE66D', label='HEADER'),
        patches.Patch(color='#CCCCCC', label='OTHER'),
    ]
    ax.legend(handles=legend_items, loc='lower right', fontsize=10)
    
    plt.tight_layout()
    return fig

fig = visualize_sample(funsd['train'][0], ID2LABEL, LABEL_COLORS, "FUNSD Training Sample — Ground Truth Labels")
plt.savefig('funsd_ground_truth.png', dpi=100, bbox_inches='tight')
plt.show()
print("Visualization saved.")

In [ ]:
MODEL_NAME = "microsoft/layoutlmv3-base"

print(f"Loading LayoutLMv3 processor from {MODEL_NAME}...")
processor = LayoutLMv3Processor.from_pretrained(
    MODEL_NAME,
    apply_ocr=False   # We provide our own words + boxes (from FUNSD / PyMuPDF)
)

print("Processor loaded.")
print(f"  Max sequence length: {processor.tokenizer.model_max_length}")
print(f"  Vocab size: {processor.tokenizer.vocab_size}")

In [ ]:
class FUNSDDataset(Dataset):
    """
    Custom PyTorch Dataset for FUNSD document layout understanding.
    
    Key design decision: LayoutLMv3 needs words + boxes + image all together.
    The processor handles tokenization and feature extraction in one call.
    
    Subword alignment: When a word is split into N subword tokens,
    only the first token gets the true label. The rest get -100 (ignored).
    """
    
    def __init__(self, hf_dataset, processor, label2id, max_length=512):
        self.dataset  = hf_dataset
        self.processor = processor
        self.label2id  = label2id
        self.max_length = max_length
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        sample = self.dataset[idx]
        
        image      = sample['image'].convert('RGB')
        words      = sample['tokens']
        boxes      = sample['bboxes']
        ner_tags   = sample['ner_tags']
        word_labels = [self.label2id.get(
            self.dataset.features['ner_tags'].feature.int2str(tag), 0
        ) for tag in ner_tags]
        
        # Process image + words + boxes together
        # apply_ocr=False means we provide our own OCR output
        encoding = self.processor(
            image,
            words,
            boxes=boxes,
            word_labels=word_labels,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        # Squeeze batch dimension from processor output
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'bbox':           encoding['bbox'].squeeze(0),
            'pixel_values':   encoding['pixel_values'].squeeze(0),
            'labels':         encoding['labels'].squeeze(0)
        }

print("FUNSDDataset class defined.")

# Build datasets and dataloaders
MAX_LENGTH = 512
BATCH_SIZE = 2    # LayoutLMv3 is large — keep batch size small on T4

train_dataset = FUNSDDataset(funsd['train'], processor, LABEL2ID, MAX_LENGTH)
test_dataset  = FUNSDDataset(funsd['test'],  processor, LABEL2ID, MAX_LENGTH)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples:  {len(test_dataset)}")

# Verify one item
item = train_dataset[0]
print(f"\nSample shapes:")
for key, val in item.items():
    print(f"  {key}: {val.shape}")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
print(f"Loading LayoutLMv3 model: {MODEL_NAME}")

model = LayoutLMv3ForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

model = model.to(device)

total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model loaded.")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Output labels:        {NUM_LABELS}")


In [ ]:
seqeval_metric = evaluate.load("seqeval")

def compute_metrics(model, dataloader, id2label, device, max_batches=None):
    """
    Compute seqeval F1 metrics for token classification.
    """
    model.eval()
    all_true_labels = []
    all_predictions = []
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            if max_batches and batch_idx >= max_batches:
                break
            
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            bbox           = batch['bbox'].to(device)
            pixel_values   = batch['pixel_values'].to(device)
            labels         = batch['labels']   # Keep on CPU for metric computation
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                bbox=bbox,
                pixel_values=pixel_values
            )
            
            # Get predicted label IDs
            logits = outputs.logits  # [batch, seq_len, num_labels]
            predictions = torch.argmax(logits, dim=-1).cpu()  # [batch, seq_len]
            
            # Convert to label strings, filtering out -100 (padding)
            for pred_seq, true_seq in zip(predictions, labels):
                pred_labels_str = []
                true_labels_str = []
                
                for pred_id, true_id in zip(pred_seq, true_seq):
                    if true_id.item() == -100:
                        continue  # Skip padding tokens
                    pred_labels_str.append(id2label[pred_id.item()])
                    true_labels_str.append(id2label[true_id.item()])
                
                all_predictions.append(pred_labels_str)
                all_true_labels.append(true_labels_str)
    
    results = seqeval_metric.compute(
        predictions=all_predictions,
        references=all_true_labels
    )
    
    return results

print("Evaluation function defined.")


In [ ]:
print("Computing baseline F1 (before fine-tuning, first 20 batches)...")

baseline_results = compute_metrics(
    model, test_loader, ID2LABEL, device, max_batches=20
)

baseline_f1        = baseline_results.get('overall_f1', 0.0)
baseline_precision = baseline_results.get('overall_precision', 0.0)
baseline_recall    = baseline_results.get('overall_recall', 0.0)

print(f"\nBaseline Results (no fine-tuning):")
print(f"  Overall F1:        {baseline_f1:.4f}")
print(f"  Overall Precision: {baseline_precision:.4f}")
print(f"  Overall Recall:    {baseline_recall:.4f}")


In [ ]:
NUM_EPOCHS    = 10
LEARNING_RATE = 5e-5
WEIGHT_DECAY  = 0.01
GRAD_CLIP     = 1.0
EVAL_BATCHES  = 30
SAVE_DIR      = "/content/layoutlmv3_funsd_finetuned"
os.makedirs(SAVE_DIR, exist_ok=True)

# Differential learning rates:
# Encoder layers (already pre-trained) → lower LR
# Classifier head (randomly initialized) → higher LR
optimizer = AdamW([
    {'params': model.layoutlmv3.parameters(), 'lr': LEARNING_RATE},
    {'params': model.classifier.parameters(), 'lr': LEARNING_RATE * 5}
], weight_decay=WEIGHT_DECAY)

# Linear warmup scheduler
total_steps  = NUM_EPOCHS * len(train_loader)
warmup_steps = int(0.1 * total_steps)
scheduler = LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=warmup_steps
)

print(f"Training config:")
print(f"  Epochs:       {NUM_EPOCHS}")
print(f"  Batch size:   {BATCH_SIZE}")
print(f"  LR (encoder): {LEARNING_RATE}")
print(f"  LR (head):    {LEARNING_RATE * 5}")
print(f"  Total steps:  {total_steps}")
print(f"  Warmup steps: {warmup_steps}")

history = {'train_loss': [], 'val_f1': []}
best_f1 = 0.0
best_epoch = -1

print("Starting fine-tuning...\n")

for epoch in range(NUM_EPOCHS):
    print(f"{'='*60}")
    print(f"EPOCH {epoch + 1} / {NUM_EPOCHS}")
    print(f"{'='*60}")
    
    # ---- TRAINING ----
    model.train()
    epoch_loss  = 0.0
    num_batches = 0
    
    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        bbox           = batch['bbox'].to(device)
        pixel_values   = batch['pixel_values'].to(device)
        labels         = batch['labels'].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            bbox=bbox,
            pixel_values=pixel_values,
            labels=labels
        )
        
        loss = outputs.loss
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        
        if epoch == 0 and num_batches < warmup_steps:
            scheduler.step()
        
        epoch_loss  += loss.item()
        num_batches += 1
    
    avg_loss = epoch_loss / num_batches
    history['train_loss'].append(avg_loss)
    print(f"  Avg Training Loss: {avg_loss:.4f}")
    
    # ---- EVALUATION ----
    print(f"  Evaluating (first {EVAL_BATCHES} batches)...")
    results = compute_metrics(model, test_loader, ID2LABEL, device, max_batches=EVAL_BATCHES)
    
    val_f1        = results.get('overall_f1', 0.0)
    val_precision = results.get('overall_precision', 0.0)
    val_recall    = results.get('overall_recall', 0.0)
    
    history['val_f1'].append(val_f1)
    
    print(f"  F1:        {val_f1:.4f}")
    print(f"  Precision: {val_precision:.4f}")
    print(f"  Recall:    {val_recall:.4f}")
    
    if val_f1 > best_f1:
        best_f1    = val_f1
        best_epoch = epoch + 1
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        print(f"  ✓ New best model saved! F1 = {best_f1:.4f}")

print(f"\nTraining complete!")
print(f"Best F1: {best_f1:.4f} at Epoch {best_epoch}")


In [ ]:
epochs = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, history['train_loss'], 'b-o', linewidth=2, markersize=6)
ax1.set_title('Training Loss', fontsize=14, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history['val_f1'], 'g-o', linewidth=2, markersize=6, label='Val F1')
ax2.axhline(y=baseline_f1, color='gray', linestyle='--',
            label=f'Baseline F1 ({baseline_f1:.3f})', alpha=0.7)
ax2.set_title('Validation F1 (higher = better)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1 Score')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('LayoutLMv3 Fine-tuning on FUNSD', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('layoutlm_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved training_curves.png")

In [ ]:
print("Loading best checkpoint...")
best_model = LayoutLMv3ForTokenClassification.from_pretrained(SAVE_DIR).to(device)
best_processor = LayoutLMv3Processor.from_pretrained(SAVE_DIR, apply_ocr=False)

print("Running final evaluation on full test set...")
final_results = compute_metrics(best_model, test_loader, ID2LABEL, device)

final_f1        = final_results.get('overall_f1', 0.0)
final_precision = final_results.get('overall_precision', 0.0)
final_recall    = final_results.get('overall_recall', 0.0)

improvement = ((final_f1 - baseline_f1) / max(baseline_f1, 1e-6)) * 100

print(f"\n{'='*55}")
print(f"FINAL BENCHMARK RESULTS")
print(f"{'='*55}")
print(f"{'Metric':<25} {'Baseline':>10} {'Fine-tuned':>12}")
print(f"{'-'*55}")
print(f"{'Overall F1':<25} {baseline_f1:>10.4f} {final_f1:>12.4f}")
print(f"{'Overall Precision':<25} {baseline_precision:>10.4f} {final_precision:>12.4f}")
print(f"{'Overall Recall':<25} {baseline_recall:>10.4f} {final_recall:>12.4f}")
print(f"{'-'*55}")
print(f"{'F1 Improvement':<25} {'—':>10} {improvement:>+11.1f}%")
print(f"{'='*55}")

# Per-class breakdown
print(f"\nPer-Class F1:")
for entity in ['QUESTION', 'ANSWER', 'HEADER']:
    f1_score = final_results.get(entity, {}).get('f1', 0.0)
    prec     = final_results.get(entity, {}).get('precision', 0.0)
    rec      = final_results.get(entity, {}).get('recall', 0.0)
    print(f"  {entity:<12} — F1: {f1_score:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}")

In [ ]:
rows = []

# Overall row
rows.append({
    'Entity':    'Overall',
    'Precision': f"{final_precision:.4f}",
    'Recall':    f"{final_recall:.4f}",
    'F1':        f"{final_f1:.4f}",
    'vs Baseline': f"{improvement:+.1f}%"
})

# Per-class rows
for entity in ['QUESTION', 'ANSWER', 'HEADER']:
    e = final_results.get(entity, {})
    rows.append({
        'Entity':    entity,
        'Precision': f"{e.get('precision', 0.0):.4f}",
        'Recall':    f"{e.get('recall', 0.0):.4f}",
        'F1':        f"{e.get('f1', 0.0):.4f}",
        'vs Baseline': '—'
    })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))
results_df.to_csv('benchmark_results.csv', index=False)
print("\nSaved benchmark_results.csv")


In [ ]:
def predict_layout(sample, model, processor, id2label, device):
    """
    Run layout classification on a single FUNSD sample.
    Returns predicted label for each word.
    """
    model.eval()
    
    image = sample['image'].convert('RGB')
    words = sample['tokens']
    boxes = sample['bboxes']
    
    encoding = processor(
        image, words, boxes=boxes,
        return_tensors='pt',
        truncation=True,
        max_length=512
    )
    
    with torch.no_grad():
        outputs = model(
            input_ids=encoding['input_ids'].to(device),
            attention_mask=encoding['attention_mask'].to(device),
            bbox=encoding['bbox'].to(device),
            pixel_values=encoding['pixel_values'].to(device)
        )
    
    # Get predicted label per token
    predictions = torch.argmax(outputs.logits, dim=-1).squeeze(0).cpu()
    
    # Map tokens back to words (use first token per word)
    word_ids   = encoding.word_ids(batch_index=0)
    pred_labels = []
    prev_word_id = None
    
    for token_idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        if word_id != prev_word_id:
            pred_labels.append(id2label[predictions[token_idx].item()])
        prev_word_id = word_id
    
    # Pad or trim to match word count
    pred_labels = pred_labels[:len(words)]
    while len(pred_labels) < len(words):
        pred_labels.append('O')
    
    return pred_labels

def draw_predictions(sample, pred_labels, label_colors, title):
    image = sample['image'].convert('RGB')
    draw  = ImageDraw.Draw(image, 'RGBA')
    
    for box, label in zip(sample['bboxes'], pred_labels):
        color = label_colors.get(label, '#CCCCCC')
        r, g, b = int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)
        x0, y0, x1, y1 = box
        draw.rectangle([x0, y0, x1, y1], fill=(r, g, b, 110), outline=(r, g, b, 230), width=2)
    
    return image

# Show side-by-side: Ground Truth vs Predicted
fig, axes = plt.subplots(2, 3, figsize=(18, 20))

for col in range(3):
    sample = funsd['test'][col]
    
    # Ground truth labels
    gt_labels = [
        funsd['test'].features['ner_tags'].feature.int2str(tag)
        for tag in sample['ner_tags']
    ]
    gt_image = draw_predictions(sample, gt_labels, LABEL_COLORS, "")
    
    # Predicted labels
    pred_labels = predict_layout(sample, best_model, best_processor, ID2LABEL, device)
    pred_image  = draw_predictions(sample, pred_labels, LABEL_COLORS, "")
    
    axes[0][col].imshow(gt_image)
    axes[0][col].set_title(f"Ground Truth (sample {col+1})", fontsize=11, fontweight='bold')
    axes[0][col].axis('off')
    
    axes[1][col].imshow(pred_image)
    axes[1][col].set_title(f"Model Prediction (sample {col+1})", fontsize=11, fontweight='bold')
    axes[1][col].axis('off')

# Legend
legend_items = [
    patches.Patch(color='#FF6B6B', label='QUESTION'),
    patches.Patch(color='#4ECDC4', label='ANSWER'),
    patches.Patch(color='#FFE66D', label='HEADER'),
    patches.Patch(color='#CCCCCC', label='OTHER'),
]
fig.legend(handles=legend_items, loc='lower center', ncol=4, fontsize=12)
plt.suptitle('LayoutLMv3: Ground Truth vs Model Predictions', fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig('predictions_vs_ground_truth.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved predictions_vs_ground_truth.png")

In [ ]:
def extract_layout_from_pdf(pdf_path):
    """
    Extract text and bounding boxes from a PDF using PyMuPDF.
    
    Returns:
        List of page dicts, each containing:
          - 'image': PIL.Image of the page
          - 'words': list of word strings
          - 'boxes': list of [x0, y0, x1, y1] normalized to [0, 1000]
          - 'page_width': original page width in points
          - 'page_height': original page height in points
    """
    doc = fitz.open(pdf_path)
    pages_data = []
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        page_width  = page.rect.width
        page_height = page.rect.height
        
        # Render page to image at 150 DPI
        mat   = fitz.Matrix(150 / 72, 150 / 72)
        pix   = page.get_pixmap(matrix=mat)
        image = Image.frombytes('RGB', [pix.width, pix.height], pix.samples)
        
        # Extract words with their bounding boxes
        word_blocks = page.get_text("words")  # Returns (x0, y0, x1, y1, word, block_no, line_no, word_no)
        
        words = []
        boxes = []
        
        for block in word_blocks:
            x0, y0, x1, y1, word = block[:5]
            
            if not word.strip():
                continue
            
            # Normalize bounding box to [0, 1000] range (LayoutLM requirement)
            norm_x0 = int((x0 / page_width)  * 1000)
            norm_y0 = int((y0 / page_height) * 1000)
            norm_x1 = int((x1 / page_width)  * 1000)
            norm_y1 = int((y1 / page_height) * 1000)
            
            # Clamp to valid range
            norm_x0 = max(0, min(1000, norm_x0))
            norm_y0 = max(0, min(1000, norm_y0))
            norm_x1 = max(0, min(1000, norm_x1))
            norm_y1 = max(0, min(1000, norm_y1))
            
            # Ensure x1 > x0 and y1 > y0
            if norm_x1 <= norm_x0: norm_x1 = norm_x0 + 1
            if norm_y1 <= norm_y0: norm_y1 = norm_y0 + 1
            
            words.append(word)
            boxes.append([norm_x0, norm_y0, norm_x1, norm_y1])
        
        pages_data.append({
            'image':       image,
            'words':       words,
            'boxes':       boxes,
            'page_width':  page_width,
            'page_height': page_height
        })
    
    doc.close()
    return pages_data

print("PyMuPDF extraction function defined.")

def classify_document_layout(page_data, model, processor, id2label, device, batch_size=512):
    """
    Run LayoutLMv3 classification on an extracted page.
    Handles documents longer than max_length by chunking.
    
    Returns:
        List of dicts: [{word, box, label, confidence}, ...]
    """
    model.eval()
    
    image = page_data['image']
    words = page_data['words']
    boxes = page_data['boxes']
    
    if not words:
        return []
    
    # Process in chunks if needed (long documents)
    chunk_size = 50  # words per chunk
    all_labels  = []
    all_confs   = []
    
    for start in range(0, len(words), chunk_size):
        chunk_words = words[start:start + chunk_size]
        chunk_boxes = boxes[start:start + chunk_size]
        
        encoding = processor(
            image,
            chunk_words,
            boxes=chunk_boxes,
            return_tensors='pt',
            truncation=True,
            max_length=512
        )
        
        with torch.no_grad():
            outputs = model(
                input_ids=encoding['input_ids'].to(device),
                attention_mask=encoding['attention_mask'].to(device),
                bbox=encoding['bbox'].to(device),
                pixel_values=encoding['pixel_values'].to(device)
            )
        
        logits      = outputs.logits.squeeze(0)
        probs       = torch.softmax(logits, dim=-1).cpu()
        predictions = torch.argmax(probs, dim=-1).cpu()
        
        # Map tokens back to words
        word_ids     = encoding.word_ids(batch_index=0)
        prev_word_id = None
        chunk_labels = []
        chunk_confs  = []
        
        for token_idx, word_id in enumerate(word_ids):
            if word_id is None:
                continue
            if word_id != prev_word_id:
                pred_id = predictions[token_idx].item()
                conf    = probs[token_idx][pred_id].item()
                chunk_labels.append(id2label[pred_id])
                chunk_confs.append(round(conf, 3))
            prev_word_id = word_id
        
        # Align with chunk word count
        chunk_labels = chunk_labels[:len(chunk_words)]
        chunk_confs  = chunk_confs[:len(chunk_words)]
        while len(chunk_labels) < len(chunk_words):
            chunk_labels.append('O')
            chunk_confs.append(0.0)
        
        all_labels.extend(chunk_labels)
        all_confs.extend(chunk_confs)
    
    # Build structured output
    results = []
    for word, box, label, conf in zip(words, boxes, all_labels, all_confs):
        results.append({
            'word':       word,
            'box':        box,
            'label':      label,
            'confidence': conf
        })
    
    return results

print("Layout classification function defined.")

def parse_document_to_json(page_results, page_num=0):
    """
    Convert layout classification output to clean structured JSON.
    Groups consecutive words with the same entity type into regions.
    """
    regions = []
    
    if not page_results:
        return {'page': page_num, 'regions': []}
    
    # Group consecutive tokens of the same entity type
    current_entity = None
    current_words  = []
    current_boxes  = []
    current_confs  = []
    
    def flush_entity():
        if not current_words or current_entity in (None, 'O'):
            return
        
        # Merge bounding boxes
        all_x0 = [b[0] for b in current_boxes]
        all_y0 = [b[1] for b in current_boxes]
        all_x1 = [b[2] for b in current_boxes]
        all_y1 = [b[3] for b in current_boxes]
        merged_box = [min(all_x0), min(all_y0), max(all_x1), max(all_y1)]
        
        # Strip BIO prefix (B-QUESTION -> QUESTION)
        entity_type = current_entity.split('-', 1)[-1] if '-' in current_entity else current_entity
        
        regions.append({
            'type':         entity_type,
            'text':         ' '.join(current_words),
            'bounding_box': merged_box,
            'confidence':   round(sum(current_confs) / len(current_confs), 3)
        })
    
    for item in page_results:
        label = item['label']
        
        # Normalize BIO: B- starts new entity, I- continues
        if label.startswith('B-') or label == 'O':
            flush_entity()
            current_entity = label
            current_words  = [item['word']]
            current_boxes  = [item['box']]
            current_confs  = [item['confidence']]
        elif label.startswith('I-'):
            current_words.append(item['word'])
            current_boxes.append(item['box'])
            current_confs.append(item['confidence'])
        else:
            flush_entity()
            current_entity = 'O'
            current_words  = []
            current_boxes  = []
            current_confs  = []
    
    flush_entity()
    
    return {'page': page_num, 'regions': regions}

print("JSON parser function defined.")

In [ ]:
# Demo - Create PDF & Run Pipeline
def create_demo_pdf(output_path):
    """
    Create a sample form-like PDF for demonstration.
    """
    doc  = fitz.open()
    page = doc.new_page(width=595, height=842)  # A4 size
    
    # Header
    page.insert_text((50, 60),  "EMPLOYEE INFORMATION FORM",     fontsize=18, fontname="helv-bold")
    page.insert_text((50, 85),  "Department of Human Resources",  fontsize=12, fontname="helv")
    
    # Horizontal line
    page.draw_line((50, 100), (545, 100))
    
    # Section 1
    page.insert_text((50, 130),  "PERSONAL DETAILS",          fontsize=14, fontname="helv-bold")
    page.insert_text((50, 160),  "Full Name:",                 fontsize=11, fontname="helv")
    page.insert_text((200, 160), "John Alexander Smith",       fontsize=11, fontname="helv")
    page.insert_text((50, 185),  "Date of Birth:",             fontsize=11, fontname="helv")
    page.insert_text((200, 185), "15 March 1990",              fontsize=11, fontname="helv")
    page.insert_text((50, 210),  "Employee ID:",               fontsize=11, fontname="helv")
    page.insert_text((200, 210), "EMP-2024-00342",             fontsize=11, fontname="helv")
    page.insert_text((50, 235),  "Nationality:",               fontsize=11, fontname="helv")
    page.insert_text((200, 235), "Indian",                     fontsize=11, fontname="helv")
    
    # Section 2
    page.insert_text((50, 275),  "CONTACT INFORMATION",        fontsize=14, fontname="helv-bold")
    page.insert_text((50, 305),  "Email Address:",              fontsize=11, fontname="helv")
    page.insert_text((200, 305), "john.smith@company.com",      fontsize=11, fontname="helv")
    page.insert_text((50, 330),  "Phone Number:",               fontsize=11, fontname="helv")
    page.insert_text((200, 330), "+91 98765 43210",             fontsize=11, fontname="helv")
    page.insert_text((50, 355),  "Residential Address:",        fontsize=11, fontname="helv")
    page.insert_text((200, 355), "42 MG Road, Indore, MP 452001", fontsize=11, fontname="helv")
    
    # Section 3
    page.insert_text((50, 395),  "EMPLOYMENT DETAILS",          fontsize=14, fontname="helv-bold")
    page.insert_text((50, 425),  "Department:",                 fontsize=11, fontname="helv")
    page.insert_text((200, 425), "Artificial Intelligence",     fontsize=11, fontname="helv")
    page.insert_text((50, 450),  "Designation:",                fontsize=11, fontname="helv")
    page.insert_text((200, 450), "ML Engineer",                 fontsize=11, fontname="helv")
    page.insert_text((50, 475),  "Date of Joining:",            fontsize=11, fontname="helv")
    page.insert_text((200, 475), "01 January 2024",             fontsize=11, fontname="helv")
    page.insert_text((50, 500),  "Salary Grade:",               fontsize=11, fontname="helv")
    page.insert_text((200, 500), "Grade A3",                    fontsize=11, fontname="helv")
    
    # Section 4
    page.insert_text((50, 540),  "EMERGENCY CONTACT",           fontsize=14, fontname="helv-bold")
    page.insert_text((50, 570),  "Contact Name:",               fontsize=11, fontname="helv")
    page.insert_text((200, 570), "Priya Smith",                 fontsize=11, fontname="helv")
    page.insert_text((50, 595),  "Relationship:",               fontsize=11, fontname="helv")
    page.insert_text((200, 595), "Spouse",                      fontsize=11, fontname="helv")
    page.insert_text((50, 620),  "Contact Phone:",              fontsize=11, fontname="helv")
    page.insert_text((200, 620), "+91 87654 32109",             fontsize=11, fontname="helv")
    
    # Footer
    page.draw_line((50, 760), (545, 760))
    page.insert_text((50, 775),  "This form is confidential. For HR use only.", fontsize=9, fontname="helv")
    page.insert_text((400, 775), "Page 1 of 1",                fontsize=9, fontname="helv")
    
    doc.save(output_path)
    doc.close()
    print(f"Demo PDF created: {output_path}")

PDF_PATH = "/content/demo_form.pdf"
create_demo_pdf(PDF_PATH)

# Run the complete pipeline on the demo PDF
print("Running full pipeline on demo PDF...\n")

# Step 1: Extract layout with PyMuPDF
print("Step 1: Extracting text + bounding boxes with PyMuPDF...")
pages_data = extract_layout_from_pdf(PDF_PATH)
print(f"  Pages extracted: {len(pages_data)}")
print(f"  Words on page 1: {len(pages_data[0]['words'])}")

# Step 2: Classify layout with LayoutLMv3
print("\nStep 2: Classifying regions with LayoutLMv3...")
page_results = classify_document_layout(
    pages_data[0], best_model, best_processor, ID2LABEL, device
)
print(f"  Classified {len(page_results)} word regions.")

# Step 3: Convert to structured JSON
print("\nStep 3: Parsing to structured JSON...")
structured_output = parse_document_to_json(page_results, page_num=0)

# Save JSON
with open('parsed_document.json', 'w') as f:
    json.dump(structured_output, f, indent=2)

print(f"  Regions found: {len(structured_output['regions'])}")
print("\nStructured JSON output:")
print(json.dumps(structured_output, indent=2))

# Visualize the pipeline output on the PDF page
def visualize_pipeline_output(page_data, page_results, label_colors):
    image = page_data['image'].copy()
    draw  = ImageDraw.Draw(image, 'RGBA')
    
    pw = page_data['page_width']
    ph = page_data['page_height']
    iw, ih = image.size
    
    for item in page_results:
        label = item['label']
        color = label_colors.get(label, '#CCCCCC')
        r, g, b = int(color[1:3], 16), int(color[3:5], 16), int(color[5:7], 16)
        
        # Denormalize boxes from [0,1000] back to image pixel coordinates
        nx0, ny0, nx1, ny1 = item['box']
        x0 = int(nx0 / 1000 * iw)
        y0 = int(ny0 / 1000 * ih)
        x1 = int(nx1 / 1000 * iw)
        y1 = int(ny1 / 1000 * ih)
        
        draw.rectangle([x0, y0, x1, y1], fill=(r, g, b, 100), outline=(r, g, b, 230), width=2)
    
    return image

fig, axes = plt.subplots(1, 2, figsize=(16, 10))

# Original PDF page
axes[0].imshow(pages_data[0]['image'])
axes[0].set_title("Original PDF (PyMuPDF render)", fontsize=12, fontweight='bold')
axes[0].axis('off')

# Classified output
classified_image = visualize_pipeline_output(pages_data[0], page_results, LABEL_COLORS)
axes[1].imshow(classified_image)
axes[1].set_title("LayoutLMv3 Classification Output", fontsize=12, fontweight='bold')
axes[1].axis('off')

legend_items = [
    patches.Patch(color='#FF6B6B', label='QUESTION (form fields)'),
    patches.Patch(color='#4ECDC4', label='ANSWER (filled values)'),
    patches.Patch(color='#FFE66D', label='HEADER (section titles)'),
    patches.Patch(color='#CCCCCC', label='OTHER'),
]
fig.legend(handles=legend_items, loc='lower center', ncol=4, fontsize=11)
plt.suptitle('Full Pipeline: PDF → PyMuPDF → LayoutLMv3 → Structured JSON',
             fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('full_pipeline_output.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved full_pipeline_output.png")

In [ ]:
print("\n" + "="*60)
print("LAYOUT-AWARE DOCUMENT PARSER — FINAL SUMMARY")
print("="*60)
print(f"  Model:    microsoft/layoutlmv3-base")
print(f"  Dataset:  FUNSD (Form Understanding in Noisy Scanned Docs)")
print(f"  Task:     Token classification (QUESTION / ANSWER / HEADER / O)")
print(f"")
print(f"  Baseline F1 (no fine-tuning): {baseline_f1:.4f}")
print(f"  Fine-tuned F1 (test set):     {final_f1:.4f}")
print(f"  F1 Improvement:               {improvement:+.1f}%")
print(f"")
print(f"  Per-class breakdown:")
for entity in ['QUESTION', 'ANSWER', 'HEADER']:
    f1 = final_results.get(entity, {}).get('f1', 0.0)
    print(f"    {entity:<12}  F1: {f1:.4f}")
print(f"")
print(f"  Outputs saved:")
print(f"    - layoutlmv3_funsd_finetuned/  (model checkpoint)")
print(f"    - benchmark_results.csv")
print(f"    - parsed_document.json         (pipeline output)")
print(f"    - full_pipeline_output.png")
print(f"    - predictions_vs_ground_truth.png")
print(f"    - layoutlm_training_curves.png")
print("="*60)